# Exercises

:::{admonition} How to use these exercises
:class: note
Attempt each from scratch in the empty code cell; the distributed hand-out omits the solutions. Import geopandas as `gpd`, keep track of the crs at every step, and measure only after reprojecting to a metric crs.
:::

## Exercise 1: Build a GeoDataFrame

Create a GeoDataFrame of two points, `A` at (7.4, 46.9) and `B` at (8.5, 47.4), in EPSG:4326. Print the epsg code and the geometry column.

In [ ]:
# Your solution here

## Exercise 2: Write and read GeoJSON

Write the GeoDataFrame from exercise 1 to `_files/points.geojson`, read it back, and confirm the shape and that the crs is preserved.

In [ ]:
# Your solution here

## Exercise 3: Reproject

Reproject the points to EPSG:2056 and print the new epsg code and the projected coordinates of point `A` (in metres, rounded to the nearest metre).

In [ ]:
# Your solution here

## Exercise 4: Measure correctly

Compute the distance between `A` and `B` in kilometres. Reproject to EPSG:2056 first, and state in a comment why measuring in EPSG:4326 would be wrong.

In [ ]:
# Your solution here

## Exercise 5: Spatial join

Given the polygon below (EPSG:4326), use `sjoin` with the `within` predicate to find which of the two points lie inside it.

```python
from shapely.geometry import Polygon
poly = gpd.GeoDataFrame({"zone": ["z"]},
    geometry=[Polygon([(7, 46.5), (8, 46.5), (8, 47.5), (7, 47.5)])], crs="EPSG:4326")
```

In [ ]:
# Your solution here

## Exercise 6: Buffer and area

Reproject the points to EPSG:2056, buffer each by 10 km, and print the area of one buffer in km² (it should be close to the analytical value pi times 10² = 314 km²).

In [ ]:
# Your solution here

## Exercise 7: Dissolve

Create a GeoDataFrame of two adjacent polygons that share the attribute `type = "flood"`, then `dissolve` by that attribute and confirm the result is a single merged polygon.

In [ ]:
# Your solution here

## Exercise 8: Hurricane track analysis

```{figure} _static/nasa-hurricane-florence-2018-09-11.jpg
---
name: hurricane-florence
width: 500px
alt: A satellite view of Hurricane Florence over the Atlantic, a clear eye at the centre of a broad spiral of cloud
---
Hurricane Florence on 11 September 2018, three days before landfall in North Carolina. Image from <a href="https://worldview.earthdata.nasa.gov/">NASA Worldview</a> via <a href="https://commons.wikimedia.org/wiki/File:Florence_2018-09-11_1750Z.jpg">Wikimedia Commons</a>, public domain.
```

*Can you quickly find out which US states Hurricane Florence passed through using geopandas?*

Apply geopandas to read in the geospatial data, plot, and analyse the track of Hurricane Florence
from 30 August to 18 September 2018. The track is the archived advisory record from the
[National Hurricane Center](https://www.nhc.noaa.gov/); the state boundaries are a
[US Census Bureau](https://www.census.gov/geographies/mapping-files/time-series/geo/carto-boundary-file.html)
cartographic boundary file.

References:

1. [Introduction to GeoPandas](https://geopandas.org/en/stable/getting_started/introduction.html) — geopandas' official website
2. [Geopandas: an introduction](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-2/geopandas-an-introduction.html) — Automating GIS Processes
3. [Use Data for Earth and Environmental Science in Open Source Python](https://www.earthdatascience.org/courses/use-data-open-source-python/)
4. [The Shapely User Manual](https://shapely.readthedocs.io/en/stable/manual.html)
5. [Geospatial Analysis with Python and R](https://kodu.ut.ee/~kmoch/geopython2020/index.html)

In [ ]:
# Pre-supplied: download and cache the two real data files.
import pooch

states_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/gz_2010_us_040_00_5m.json",
    known_hash="sha256:7a8c022e063a34a83f35984cde6c81992ece5983f8cc4459ed02e40687739573",
    fname="us_states.geojson",
    path=pooch.os_cache("mlees"),
)
track_path = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/florence.csv",
    known_hash="sha256:385691583ee41682a1c905e042c56ea362609fb04645934bdda7911c55c8b63f",
    fname="florence.csv",
    path=pooch.os_cache("mlees"),
)

:::{admonition} Beyond this subchapter
:class: note
Two of the tools below were not demonstrated in 1.6's lecture, and are linked where they are
needed:

- [`gpd.points_from_xy`](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html) — Q5, for turning two coordinate columns into a geometry column
- [`GeoDataFrame.annotate` via matplotlib's `ax.annotate`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html) — Q8, for labelling each state; `ax.annotate` itself is from 1.4

Everything else is from the lecture: `read_file`, `.plot()`, `GeoDataFrame`, `.crs`, `geom_type`,
`overlay`, `centroid`, `to_crs`, `buffer` and the spatial predicates. `.head()` and `.isin()` are
from 1.5.
:::

**Q1) Import geopandas and pandas.**

Import geopandas as `gpd`, pandas as `pd`, and matplotlib's pyplot as `plt`.

In [ ]:
# Import geopandas (as gpd), pandas, and matplotlib

**Q2) Read the state boundary file with geopandas' `read_file` function.**

The file is cached at `states_path`. Call the result `country`.

In [ ]:
# Read the data with the geopandas function

**Q3) Have a look at the data. What type of geometries does it contain?**

Print the first few rows, and answer the geometry question in a comment.

In [ ]:
# Print out the first few lines of the data

**Q4) Have a look at the data on a map using geopandas' `.plot()` method.**

Exclude Alaska and Hawaii using the `NAME` attribute and pandas' `.isin()` method. Specify the
figsize to be 30 x 20.

In [ ]:
# Plot the US states (Alaska and Hawaii excluded)

In [ ]:
# Pre-supplied: read in the hurricane Florence data, drop the advisory bookkeeping columns,
# fix the longitude sign, and have a look at the dataframe.
# The Long column holds a positive "degrees west" magnitude, not a signed longitude -- a common
# quirk of NHC advisory data.
florence = pd.read_csv(track_path)
florence = florence.drop(["AdvisoryNumber", "Forecaster", "Received"], axis=1)
florence["Long"] = 0 - florence["Long"]
florence.head(3)

**Q5) Create a GeoDataFrame from the `florence` DataFrame.**

Build the geometry column from the `Long` and `Lat` columns with
[`gpd.points_from_xy`](https://geopandas.org/en/stable/docs/reference/api/geopandas.points_from_xy.html),
and call the result `gdf_florence`. Then look at its first few rows.

In [ ]:
# Create a geodataframe from the hurricane florence dataframe

**Q6) Plot the US states map (without Alaska and Hawaii) and hurricane Florence together.**

Draw the states as a base map, then plot the hurricane positions on top in a colour that stands
out.

In [ ]:
# Plot to see the hurricane overlay the US map, with the hurricane position on top

**Q7) What is the coordinate reference system of the data?**

Check it for both layers, and say in a comment whether they match — two layers only line up on
one pair of axes if they share a crs.

In [ ]:
# Check the coordinate reference system of the data

**Q8) Which states did the hurricane pass through?**

```{hint}
One approach is to plot the trajectory over the US map, annotate each state with its name, and
read the answer off the figure. Other approaches are more than welcome — `overlay` with
`how="intersection"` will give you the track points that fall inside a state polygon, and the
`NAME` column of the result then answers the question directly.
```

In [ ]:
# Plot the US states without Alaska and Hawaii, then
# annotate the US states with their names, then
# select the hurricane trajectory points inside the US boundary with the overlay operation, then
# plot the hurricane trajectory inside the US boundary